# 5교시. 문서 자동화 웹 애플리케이션 기본 구현

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leecks1119/document_ai_lecture/blob/master/colab/05_streamlit_basic.ipynb)

**이번 교시 행동:** 업로드·실행·원문·JSON 영역을 만들고 업로드한 파일명이 화면에 반영되는지 확인합니다.

**통과 증거:** `course_outputs/app_05.py`

> Google Colab도 외부 클라우드입니다. 조직 승인 없는 개인·회사 문서는
> 업로드하지 않습니다. 필수 실습은 저장소의 비식별 공개·합성 샘플만
> 사용합니다.

화면의 **처리 방식**을 먼저 확인합니다.

- **지금 이 사진을 직접 읽었습니다:** 현재 파일에 OCR 모델을 실행한 결과입니다.
- **수업용 예제 결과를 불러왔습니다:** 현재 파일을 분석한 결과가 아닙니다.
- 3분 이상 멈추면 실행을 중지하고 수업용 예제로 계속합니다.
- 각 교시 끝에서 `CHECKPOINT PASS`와 산출물 파일을 확인합니다.


## 이 노트북에서 내가 하는 일

- **필수 실습:** 지금까지 만든 처리 결과를 파일 업로드·실행 버튼·결과 영역이 있는 웹앱으로 감쌉니다.
- **내가 바꾸는 곳:** 앱 제목과 버튼 문구 두 곳만 업무에 맞게 바꿉니다.
- **인터넷 자료로 다시 실험:** 내가 고른 문서를 처음 보는 동료도 버튼의 행동을 이해할 수 있는지 확인합니다.

먼저 제공 샘플로 끝까지 실행해 `CHECKPOINT PASS`를 만드세요. 그다음
[공개·비식별 실습 자료 찾기](https://github.com/leecks1119/document_ai_lecture/blob/master/docs/public_practice_sources.md)를 보고
입력 한 장만 바꾸어 다시 실행합니다. 2교시에서 고른 자료와 결과 파일은
3~7교시에 그대로 이어 쓰므로 매 시간 새 자료를 찾을 필요가 없습니다.

> `🟢 그대로 실행하는 셀`은 수정하지 않습니다. `🟠 내가 짧게 바꾸는
> 셀`만 필수이고, `🔵 원하면 바꾸는 셀`은 시간이 남을 때 합니다.
> 정답은 모두 공개되어 있으므로 정답을 먼저 복사하고 결과를 관찰해도 됩니다.

## 코드 셀을 읽는 방법

각 코드 셀의 맨 위에는 `코드 읽기` 주석이 있습니다.

1. `수정하지 않습니다`라고 적힌 셀은 설명을 읽고 그대로 실행합니다.
2. 주황색 필수 `TODO`만 채웁니다. 파란색 선택 `TODO`는 건너뛰어도 됩니다.
3. 실행 출력에서 `코드 읽는 법`과 `확인할 결과`를 다시 확인합니다.
4. `단계 실행 완료`가 나온 뒤 다음 코드 셀로 이동합니다.

Python 문법 전체를 먼저 이해할 필요는 없습니다. 변수에 어떤 값이 들어가고,
실행 뒤 어떤 결과가 달라지는지를 중심으로 읽습니다.


In [ ]:
def _show_learning_message(markdown_text):
    try:
        from IPython.display import Markdown, display
        display(Markdown(markdown_text))
    except ImportError:
        print(markdown_text)


def show_lab_step(
    current,
    total,
    title,
    action,
    expected,
    code_help,
    edit_kind,
):
    cell_kind = {
        "required": "🟠 내가 짧게 바꾸는 셀",
        "optional": "🔵 원하면 바꾸는 셀",
        "none": "🟢 그대로 실행하는 셀",
    }[edit_kind]
    _show_learning_message(
        f"""---
### {cell_kind} · {current}/{total} · {title}

**지금 할 일:** {action}

**코드 읽는 법:** {code_help}

**이 단계에서 확인할 결과:** {expected}
"""
    )


def complete_lab_step(current, total, expected):
    next_action = (
        "결과를 확인한 뒤 다음 코드 셀을 실행하세요."
        if current < total
        else "마지막 CHECKPOINT와 산출물 파일을 확인하세요."
    )
    _show_learning_message(
        f"""> ✅ **{current}/{total} 단계 실행 완료**
>
> **결과 확인:** {expected}
>
> **다음 행동:** {next_action}
"""
    )

# ── 코드 읽기 ─────────────────────────────────────────────
# 웹앱 파일을 저장할 `OUTPUT_DIR`와 Colab 공통 함수를 준비합니다. 설정 코드이므로 수정하지 않습니다.
# ──────────────────────────────────────────────────────────
show_lab_step(1, 7, '공통 환경 준비', '웹앱 파일을 저장할 폴더와 실습 공통 기능을 준비합니다.', 'Python·Platform·공통 작업 폴더가 표시되어야 합니다.', '웹앱 파일을 저장할 `OUTPUT_DIR`와 Colab 공통 함수를 준비합니다. 설정 코드이므로 수정하지 않습니다.', 'none')

import json
import os
import platform
import sys
from pathlib import Path

OUTPUT_DIR = Path("course_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
VALIDATION_MODE = os.getenv("COURSE_VALIDATE_EXAMPLE") == "1"

def upload_previous_artifact(filename):
    target = OUTPUT_DIR / filename
    if target.exists() or VALIDATION_MODE:
        return target if target.exists() else None
    try:
        from google.colab import files
    except ImportError:
        return None
    print(f"이전 교시에서 내려받은 {filename}을 선택하세요.")
    uploaded = files.upload()
    if filename not in uploaded:
        raise FileNotFoundError(
            f"{filename}이 선택되지 않았습니다. 준비 입력을 쓰려면 "
            "USE_COURSE_EXAMPLE=True로 바꾸세요."
        )
    target.write_bytes(uploaded[filename])
    return target


def download_artifact(path):
    if VALIDATION_MODE:
        return
    try:
        from google.colab import files
    except ImportError:
        return
    files.download(str(path))

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("공통 작업 폴더:", OUTPUT_DIR.resolve())

COURSE_ASSET_BASE_URL = (
    "https://raw.githubusercontent.com/leecks1119/"
    "document_ai_lecture/master/"
)

def load_course_assets(*relative_paths):
    if VALIDATION_MODE:
        local_root = os.getenv("COURSE_LOCAL_ASSET_ROOT")
        if not local_root:
            raise RuntimeError(
                "자동 검증용 COURSE_LOCAL_ASSET_ROOT가 필요합니다."
            )
        root = Path(local_root)
        return {
            path: (root / path).read_bytes()
            for path in relative_paths
        }

    import requests

    loaded = {}
    missing = []
    for path in relative_paths:
        try:
            response = requests.get(
                COURSE_ASSET_BASE_URL + path,
                timeout=30,
            )
            response.raise_for_status()
            loaded[path] = response.content
        except requests.RequestException as exc:
            print(f"자동 다운로드 실패: {Path(path).name} · {exc}")
            missing.append(path)

    if missing:
        from google.colab import files

        expected = ", ".join(Path(path).name for path in missing)
        print("다음 파일을 저장소에서 내려받아 선택하세요:", expected)
        uploaded = files.upload()
        uploaded_by_name = {
            Path(name).name: content
            for name, content in uploaded.items()
        }
        for path in missing:
            filename = Path(path).name
            if filename not in uploaded_by_name:
                raise FileNotFoundError(
                    f"{filename}이 선택되지 않았습니다."
                )
            loaded[path] = uploaded_by_name[filename]

    return loaded

complete_lab_step(1, 7, 'Python·Platform·공통 작업 폴더가 표시되어야 합니다.')


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `required_streamlit`과 설치 버전을 비교합니다. 버전이 다를 때만 `pip install`을 실행하므로 이 셀은 그대로 실행합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(2, 7, '웹앱·OCR 실행환경 준비', 'Streamlit과 PaddleOCR 버전을 확인하고 필요하면 설치합니다.', '웹앱 버전과 직접 OCR 실행 준비 결과가 표시되어야 합니다.', '`required_streamlit`과 설치 버전을 비교합니다. 버전이 다를 때만 `pip install`을 실행하므로 이 셀은 그대로 실행합니다.', 'none')

import importlib.metadata
import subprocess

required_streamlit = "1.60.0"
try:
    installed_streamlit = importlib.metadata.version("streamlit")
except importlib.metadata.PackageNotFoundError:
    installed_streamlit = None
if installed_streamlit != required_streamlit:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", f"streamlit=={required_streamlit}"]
    )

complete_lab_step(2, 7, '웹앱 버전과 직접 OCR 실행 준비 결과가 표시되어야 합니다.')


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `app_code`는 Streamlit 화면의 전체 소스이고 `write_text()`가 `app_05.py`로 저장합니다. 긴 문자열은 앱 파일을
# 만드는 재료입니다.
# ──────────────────────────────────────────────────────────
show_lab_step(3, 7, '기본 웹앱 파일 생성', '업로드·실행 버튼·원문·JSON 영역이 있는 앱을 저장합니다.', '`app_05.py` 저장 경로가 표시되어야 합니다.', '`app_code`는 Streamlit 화면의 전체 소스이고 `write_text()`가 `app_05.py`로 저장합니다. 긴 문자열은 앱 파일을 만드는 재료입니다.', 'none')

app_code = (
    'import streamlit as st\n'
    '\n'
    "GOLDEN_RECEIPT = {'document_type': 'receipt',\n"
    " 'store_name': '이태리집',\n"
    " 'date': '2025-10-04',\n"
    " 'total_amount': 76000,\n"
    " 'items': [{'name': '페퍼로니 앤 치즈',\n"
    "            'quantity': 1,\n"
    "            'unit_price': 29000,\n"
    "            'line_total': 29000},\n"
    "           {'name': '토마토 파스타', 'quantity': 1, 'unit_price': 14000, 'line_total': 14000},\n"
    "           {'name': '수제 돈가스', 'quantity': 1, 'unit_price': 13000, 'line_total': 13000},\n"
    "           {'name': '새우 칠리치 필라',\n"
    "            'quantity': 1,\n"
    "            'unit_price': 14000,\n"
    "            'line_total': 14000},\n"
    "           {'name': '콜라', 'quantity': 3, 'unit_price': 2000, 'line_total': 6000}],\n"
    " 'adjustments': {'discount': 0, 'tax': 0, 'service': 0, 'rounding': 0},\n"
    " 'tax_breakdown': {'mode': 'included_in_item_prices',\n"
    "                   'supply_amount': 69094,\n"
    "                   'vat': 6906,\n"
    "                   'payable_total': 76000},\n"
    " 'raw_values': {'store_name': '이태리집',\n"
    "                'date': '2025-10-04 12:33:37',\n"
    "                'total_amount': '76,000'},\n"
    " 'cleaned_values': {'store_name': '이태리집', 'date': '2025-10-04', 'total_amount': 76000},\n"
    " 'evidence': {'store_name': {'raw_value': '이태리집', 'line': 1},\n"
    "              'date': {'raw_value': '거래일시 2025-10-04 12:33:37', 'line': 2},\n"
    "              'total_amount': {'raw_value': '합계 금액 76,000', 'line': 8}},\n"
    " 'source_mode': 'course_example_rule_extraction'}\n"
    "GOLDEN_OCR_TEXT = '이태리집\\n거래일시 2025-10-04 12:33:37\\n페퍼로니 앤 치즈 29,000 1 29,000\\n토마토 파스타 14,000 1 14,000\\n수제 돈가스 13,000 1 13,000\\n새우 칠리치 필라 14,000 1 14,000\\n콜라 2,000 3 6,000\\n합계 금액 76,000\\n부가세 과세물품가액 69,094\\n부가세 6,906\\n'\n"
    '\n'
    'st.set_page_config(page_title="영수증 Document AI", layout="wide")\n'
    'st.title("영수증 Document AI 미니 앱")\n'
    'uploaded = st.file_uploader(\n'
    '    "승인된 비식별 이미지 또는 PDF 한 장 · 최대 5MB",\n'
    '    type=["png", "jpg", "jpeg", "pdf"],\n'
    '    max_upload_size=5,\n'
    '    help="PNG, JPG, JPEG, PDF만 허용합니다. 수업에서는 한 번에 5MB 이하 한 장만 처리합니다.",\n'
    ')\n'
    'if uploaded is not None:\n'
    '    st.success(f"업로드 연결 확인: {uploaded.name} · {len(uploaded.getvalue()):,} bytes")\n'
    '    st.caption("이 파일은 6교시에서 실제 처리 함수와 연결합니다.")\n'
    '\n'
    'if st.button("수업용 예제 결과 보기"):\n'
    '    st.info(\n'
    '        "수업용 예제 결과를 불러왔습니다. "\n'
    '        "현재 업로드한 파일을 분석한 결과가 아닙니다."\n'
    '    )\n'
    '    st.text_area("판독 원문", GOLDEN_OCR_TEXT, height=220)\n'
    '    st.json(GOLDEN_RECEIPT)\n'
)

output_path = OUTPUT_DIR / "app_05.py"
output_path.write_text(app_code, encoding="utf-8")
print("저장:", output_path)

complete_lab_step(3, 7, '`app_05.py` 저장 경로가 표시되어야 합니다.')


## 내가 직접 바꾸는 화면 문구 2개

앱 제목과 실행 버튼 문구를 업무 사용자가 이해할 표현으로 바꿉니다.


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `my_app_title`과 `my_button_label` 두 `None`만 원하는 문구로 바꿉니다. Python 문법보다 사용자에게 보이는
# 표현을 설계하는 단계입니다.
# ──────────────────────────────────────────────────────────
show_lab_step(4, 7, '내 화면 문구 입력', '앱 제목과 실행 버튼을 업무 사용자가 이해할 말로 바꿉니다.', '빈칸 안내 또는 입력한 두 문구가 표시되어야 합니다.', '`my_app_title`과 `my_button_label` 두 `None`만 원하는 문구로 바꿉니다. Python 문법보다 사용자에게 보이는 표현을 설계하는 단계입니다.', 'required')

# TODO: None 두 곳을 채우세요.
my_app_title = None
my_button_label = None
if None in (my_app_title, my_button_label):
    print("빈칸이 있습니다. 아래 전체 정답과 비교하세요.")

complete_lab_step(4, 7, '빈칸 안내 또는 입력한 두 문구가 표시되어야 합니다.')


<details>
<summary>힌트와 전체 정답 보기</summary>

문서 종류와 버튼을 눌렀을 때 일어나는 일을 그대로 적습니다.
</details>


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `replace()`가 기본 제목과 버튼 문구를 내 문구로 바꾼 뒤 앱 파일을 다시 저장합니다. 빈칸이면 공개 정답을 사용합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(5, 7, '화면 문구 적용', '내 문구 또는 공개 정답을 실제 앱 파일에 반영합니다.', '적용된 앱 제목과 버튼 문구를 확인합니다.', '`replace()`가 기본 제목과 버튼 문구를 내 문구로 바꾼 뒤 앱 파일을 다시 저장합니다. 빈칸이면 공개 정답을 사용합니다.', 'none')

ANSWER_APP_TITLE = my_app_title or "영수증 검토 미니 앱"
ANSWER_BUTTON_LABEL = my_button_label or "공개 영수증 결과 확인"
app_code = app_code.replace(
    "영수증 Document AI 미니 앱",
    ANSWER_APP_TITLE,
).replace(
    "수업용 예제 결과 보기",
    ANSWER_BUTTON_LABEL,
)
output_path.write_text(app_code, encoding="utf-8")
print("내 화면 문구:", ANSWER_APP_TITLE, "/", ANSWER_BUTTON_LABEL)

complete_lab_step(5, 7, '적용된 앱 제목과 버튼 문구를 확인합니다.')


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `AppTest`는 브라우저를 열지 않고 제목·업로드·버튼·결과 영역을 검사합니다. 예외가 없고 준비 결과가 보이면 통과입니다.
# ──────────────────────────────────────────────────────────
show_lab_step(6, 7, '웹앱 자동 동작 검사', '업로드·버튼·준비 결과 화면이 실제로 동작하는지 검사합니다.', '`CHECKPOINT 1/1 PASS`가 표시되어야 합니다.', '`AppTest`는 브라우저를 열지 않고 제목·업로드·버튼·결과 영역을 검사합니다. 예외가 없고 준비 결과가 보이면 통과입니다.', 'none')

from streamlit.testing.v1 import AppTest

app_test = AppTest.from_file(str(output_path)).run(timeout=20)
assert not app_test.exception
assert app_test.title[0].value == ANSWER_APP_TITLE
assert len(app_test.file_uploader) == 1
assert len(app_test.button) == 1
app_test.button[0].click().run(timeout=20)
assert any(
    "현재 업로드한 파일을 분석한 결과가 아닙니다" in item.value
    for item in app_test.info
)
print("CHECKPOINT 1/1 PASS: 업로드·버튼·결과 화면")

complete_lab_step(6, 7, '`CHECKPOINT 1/1 PASS`가 표시되어야 합니다.')


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `subprocess.Popen()`이 Streamlit 서버를 열고 Colab iframe에 표시합니다. 선택 실습이며 화면이 안 열려도 앞의
# AppTest 결과는 유지됩니다.
# ──────────────────────────────────────────────────────────
show_lab_step(7, 7, '웹앱 직접 조작', 'Colab 안에서 Streamlit 화면을 열어 버튼과 입력을 조작합니다.', '앱 화면 또는 검증 모드 생략 안내를 확인합니다.', '`subprocess.Popen()`이 Streamlit 서버를 열고 Colab iframe에 표시합니다. 선택 실습이며 화면이 안 열려도 앞의 AppTest 결과는 유지됩니다.', 'none')

# 선택 실습 · 녹화에서는 이 셀로 실제 화면을 엽니다.
# AppTest가 필수 검증이며, 미리보기에는 공개 비식별 샘플만 사용합니다.
if not VALIDATION_MODE:
    import subprocess
    import time
    import urllib.request

    preview_process = subprocess.Popen(
        [
            sys.executable, "-m", "streamlit", "run",
            str(output_path),
            "--server.port", "8505",
            "--server.headless", "true",
            "--server.enableCORS", "false",
            "--server.enableXsrfProtection", "false",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.STDOUT,
    )
    for _ in range(20):
        try:
            urllib.request.urlopen(
                "http://127.0.0.1:8505/_stcore/health",
                timeout=1,
            )
            break
        except Exception:
            time.sleep(0.5)
    try:
        from google.colab import output
        print("아래 화면에서 직접 버튼과 입력값을 조작하세요.")
        output.serve_kernel_port_as_iframe(8505, height=760)
    except Exception as exc:
        print("Colab 미리보기를 열지 못했습니다:", exc)
        print("AppTest 결과와 app 파일로 계속합니다.")
else:
    print("검증 모드: 대화형 Streamlit 미리보기 생략")

complete_lab_step(7, 7, '앱 화면 또는 검증 모드 생략 안내를 확인합니다.')
